## Scope and AI-assistance disclosure

**Scope:** This notebook develops the Hospital 4 audit pipeline, following the same
audit/evaluation framework built for Hospital 1 in `mainHP_1_final.ipynb`. Hospital 4's
contract (`conditional_reimbursement_agreement.md`) has a different structure and section
numbering than Hospital 1's, so the contract-extraction cells are rewritten against
Hospital 4's actual clauses; the service-matching vocabulary, the pricing-order engine,
and the aggregation/confidence logic are carried over unchanged, since the underlying
billing-description abbreviations and the contractual adjustment order (bundle -> facility
-> plan -> premium/uplift -> discount) are the same across hospitals in this exercise.

**AI assistance:** Claude was used to read Hospital 1's notebook, port its reusable
service-matching and pricing/aggregation logic, and re-derive the Hospital-4-specific
contract-extraction and checks cells from `conditional_reimbursement_agreement.md`. The
final contract interpretations, thresholds, and implementation decisions were reviewed by
the author; see the decision log at the end of this notebook.

Hospital 4 has no labelled development set (only Hospital 1 does), so there is no
evaluation-against-labels section here -- the H1 notebook's dev-set metrics stand as the
calibration evidence for this shared method.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path = None, marker: str = "invoices") -> Path:
    # Works whether the notebook is opened from the repository, /mnt/data, or another Jupyter cwd.
    candidates = []
    start = start or Path.cwd()
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path("cd../insurance_auditing-main"),
        Path("/mnt/data/insurance_repo/insurance_auditing-main"),
    ])
    for candidate in candidates:
        if (candidate / marker).is_dir() and (candidate / "contracts").is_dir():
            return candidate.resolve()
    checked = ", ".join(str(c) for c in candidates[:6])
    raise FileNotFoundError(f"Could not find the insurance-auditing repository. Checked: {checked}")

REPO_ROOT = find_repo_root()
print("Repo root:", REPO_ROOT)
print("Working dir was:", Path.cwd())

invoices = pd.read_csv(
    REPO_ROOT / "invoices" / "hospital_4_invoices.csv",
    parse_dates=["invoice_date"],
)
line_items = pd.read_csv(
    REPO_ROOT / "invoices" / "hospital_4_line_items.csv",
    parse_dates=["service_date"],
)
print(f"Loaded {len(invoices)} invoices, {len(line_items)} line items")


Repo root: C:\Users\AMER\Downloads\17Sep\insurance_auditing-main
Working dir was: c:\Users\AMER\Downloads\17Sep\insurance_auditing-main
Loaded 840 invoices, 10560 line items


## Section 3 -- Base Rates, and clause 2.1 / 2.2 data checks

Clause 2.1: term 1 Jan 2024 -- 31 Dec 2025. Clause 2.2: single facility (F-MAIN), no
facility differential, and plan tier does not affect the rate. Unlike Hospital 1's
contract (clause 1.3), Hospital 4's contract does not state a closed set of valid
plan-tier values -- it only says plan tier is irrelevant to pricing -- so plan tier is
used as a pricing input to ignore, not as an audit check.


In [2]:
from decimal import Decimal
import re
import pandas as pd
from IPython.display import display

CONTRACT_NUMBER = "INS-H4-2024-2049"
CONTRACT_START = pd.Timestamp("2024-01-01")
CONTRACT_END = pd.Timestamp("2025-12-31")
FACILITY_CODE = "F-MAIN"

contract_path = REPO_ROOT / "contracts" / "hospital_4" / "conditional_reimbursement_agreement.md"
contract_lines = contract_path.read_text(encoding="utf-8").splitlines()

# ---- Section 3: Base Rates ----
rate_rows = []
section = None
for line_number, line in enumerate(contract_lines, start=1):
    if line.startswith("## 3. Base Rates"):
        section = 3
        continue
    if line.startswith("## 4."):
        section = None
    if not line.startswith("|") or section != 3:
        continue
    values = [v.strip() for v in line.strip("|").split("|")]
    if values[0] in {"Service", "---"}:
        continue
    if len(values) != 3 or not values[2].startswith("GBP"):
        continue
    base_rate_cents = int(Decimal(values[2].replace("GBP", "").replace(",", "").strip()) * 100)
    rate_rows.append({
        "service": values[0], "contract_unit": values[1],
        "base_rate_cents": base_rate_cents, "contract_section": "3", "source_line": line_number,
    })

rate_schedule = pd.DataFrame(rate_rows)
assert not rate_schedule.empty
assert rate_schedule["service"].is_unique
assert rate_schedule["base_rate_cents"].gt(0).all()

print("Contract services extracted:", len(rate_schedule))
display(rate_schedule.head(10))

# ---- Clause 2.1 term check (line-item level), without mutating raw data ----
term_check = line_items.copy()
term_check["service_date_parsed"] = pd.to_datetime(term_check["service_date"], format="%Y-%m-%d", errors="coerce")
term_check["term_issue"] = (
    term_check["service_date_parsed"].isna()
    | (term_check["service_date_parsed"] < CONTRACT_START)
    | (term_check["service_date_parsed"] > CONTRACT_END)
)
print("\nService lines checked:", len(term_check))
print("Term issues found:", int(term_check["term_issue"].sum()))
display(term_check.loc[term_check["term_issue"], ["line_id", "invoice_id", "service_date", "service_date_parsed"]])

# ---- Clause 2.2 facility check (invoice level) ----
facility_check = invoices.copy()
facility_check["facility_issue"] = facility_check["facility_code"].isna() | facility_check["facility_code"].ne(FACILITY_CODE)
print("\nInvoice records checked:", len(facility_check))
print("Facility issues found:", int(facility_check["facility_issue"].sum()))


Contract services extracted: 98


,service,contract_unit,base_rate_cents,contract_section,source_line
0,Advanced Oncology Ward Bed Occupancy,per night of occupancy,68700,3,39
1,Advanced Orthopaedic Ward Bed Occupancy,per night of occupancy,97925,3,40
2,Advanced Paediatric Theatre Time,per hour,37100,3,41
3,Advanced Palliative Home Visit,per visit,11450,3,42
4,Advanced Vascular Endoscopic Procedure,per procedure,321775,3,43
5,Ambulatory Cardiac Ward Bed Occupancy,per day of service,46350,3,44
6,Ambulatory Gastrointestinal Nutritional Support,per unit dispensed,4175,3,45
7,Ambulatory Haematology Transfusion Service,per procedure,402275,3,46
8,Ambulatory Immunologic Ward Bed Occupancy,per day of service,74700,3,47
9,Ambulatory Musculoskeletal Specimen Analysis,per item supplied,9550,3,48



Service lines checked: 10560
Term issues found: 12


,line_id,invoice_id,service_date,service_date_parsed
1003,H4-L00079-01,INV-H4-000079,2026-10-06,2026-10-06
1006,H4-L00079-04,INV-H4-000079,2023-06-14,2023-06-14
1877,H4-L00153-12,INV-H4-000153,31/02/2024,NaT
1986,H4-L00164-11,INV-H4-000164,2024-00-17,NaT
3195,H4-L00260-07,INV-H4-000260,2023-04-25,2023-04-25
5524,H4-L00445-09,INV-H4-000445,2026-10-15,2026-10-15
6543,H4-L00528-13,INV-H4-000528,2024-00-17,NaT
7200,H4-L00576-05,INV-H4-000576,2024-13-05,NaT
8514,H4-L00678-08,INV-H4-000678,2025-02-30,NaT
8697,H4-L00693-11,INV-H4-000693,2023-04-05,2023-04-05



Invoice records checked: 840
Facility issues found: 0


## Sections 5-10 -- Premiums, quantity limits, bundles, discounts, exclusions, uplifts

Hospital 4's clause numbering differs from Hospital 1's, but the substance of each
section maps directly onto Hospital 1's Sections 4-10:

| Hospital 4 | Purpose | Hospital 1 equivalent |
|---|---|---|
| Sec. 5 Threshold Premiums | aggregate per-Service-Day quantity trigger | Sec. 5 |
| Sec. 6 Daily Quantity Limits | per-patient per-day cap, 18 services | Sec. 8 |
| Sec. 7 Bundled Delivery | same-day same-patient rate substitution | Sec. 9 |
| Sec. 8 Discounts | cumulative hospital-wide utilisation, tiered | Sec. 7 |
| Sec. 9 Exclusion Windows | non-billable if paired service nearby | Sec. 10 |
| Sec. 10 Non-Business-Day Uplifts | -- | Sec. 6 |

**Section 10 is explicitly `_None._`** -- Hospital 4's agreement defines no weekend/non-
business-day uplift at all. The weekend-uplift machinery is kept (as an empty table) so
the pricing engine has the same shape as Hospital 1's, but it never fires for this
hospital.


In [3]:
from decimal import Decimal
import re
import pandas as pd
from IPython.display import display

contract_lines = contract_path.read_text(encoding="utf-8").splitlines()

# ---- Section 5: Threshold Premiums ----
threshold_rows = []
section = None
for line_number, line in enumerate(contract_lines, start=1):
    if line.startswith("## 5. Threshold Premiums"):
        section = 5; continue
    if line.startswith("## 6."):
        section = None
    if not line.startswith("|") or section != 5:
        continue
    values = [v.strip() for v in line.strip("|").split("|")]
    if values[0] in {"Service", "---"} or len(values) != 3:
        continue
    t, u = re.search(r"\d+", values[1]), re.search(r"\d+", values[2])
    if t and u:
        threshold_rows.append({"service": values[0], "threshold_units": int(t.group()),
                                "premium_percent": int(u.group()), "contract_section": "5", "source_line": line_number})
threshold_premiums = pd.DataFrame(threshold_rows)
assert len(threshold_premiums) == 18

# ---- Section 6: Daily Quantity Limits ----
cap_rows = []
section = None
for line_number, line in enumerate(contract_lines, start=1):
    if line.startswith("## 6. Daily Quantity Limits"):
        section = 6; continue
    if line.startswith("## 7."):
        section = None
    if not line.startswith("|") or section != 6:
        continue
    values = [v.strip() for v in line.strip("|").split("|")]
    if values[0] in {"Service", "---"} or len(values) != 2:
        continue
    c = re.search(r"\d+", values[1])
    if c:
        cap_rows.append({"service": values[0], "maximum_units_per_patient_day": int(c.group()),
                          "contract_section": "6", "source_line": line_number})
daily_caps = pd.DataFrame(cap_rows)
assert len(daily_caps) == 18

# ---- Section 7: Bundled Delivery ----
bundle_rows = []
section = None
for line_number, line in enumerate(contract_lines, start=1):
    if line.startswith("## 7. Bundled Delivery"):
        section = 7; continue
    if line.startswith("## 8."):
        section = None
    if not line.startswith("|") or section != 7:
        continue
    values = [v.strip() for v in line.strip("|").split("|")]
    if values[0] in {"Service A", "---"} or len(values) != 4:
        continue
    rate_a = int(Decimal(values[1].replace("GBP", "").replace(",", "").strip()) * 100)
    rate_b = int(Decimal(values[3].replace("GBP", "").replace(",", "").strip()) * 100)
    bundle_rows.append({"service_a": values[0], "bundled_rate_a_cents": rate_a,
                         "service_b": values[2], "bundled_rate_b_cents": rate_b,
                         "contract_section": "7", "source_line": line_number})
bundles = pd.DataFrame(bundle_rows)
assert len(bundles) == 7

# ---- Section 8: Discounts (cumulative, hospital-wide utilisation) ----
discount_rows = []
section = None
for line_number, line in enumerate(contract_lines, start=1):
    if line.startswith("## 8. Discounts"):
        section = 8; continue
    if line.startswith("## 9."):
        section = None
    if not line.startswith("|") or section != 8:
        continue
    values = [v.strip() for v in line.strip("|").split("|")]
    if values[0] in {"Service", "---"} or len(values) != 3:
        continue
    t = re.search(r"\((\d+)\)", values[1])
    d = re.search(r"\((\d+)%\)", values[2])
    if t and d:
        discount_rows.append({"service": values[0], "threshold_units": int(t.group(1)),
                               "discount_percent": int(d.group(1)), "contract_section": "8", "source_line": line_number})
volume_discounts = pd.DataFrame(discount_rows)
assert len(volume_discounts) == 4

# ---- Section 9: Exclusion Windows ----
exclusion_rows = []
section = None
for line_number, line in enumerate(contract_lines, start=1):
    if line.startswith("## 9. Exclusion Windows"):
        section = 9; continue
    if line.startswith("## 10."):
        section = None
    if not line.startswith("|") or section != 9:
        continue
    values = [v.strip() for v in line.strip("|").split("|")]
    if values[0] in {"Service", "---"} or len(values) != 3:
        continue
    d = re.search(r"\d+", values[1])
    if d:
        exclusion_rows.append({"service_a": values[0], "window_days": int(d.group()),
                                "service_b": values[2], "contract_section": "9", "source_line": line_number})
exclusion_windows = pd.DataFrame(exclusion_rows)
assert len(exclusion_windows) == 15

# ---- Section 10: Non-Business-Day Uplifts -- the contract states "_None._" ----
weekend_uplifts = pd.DataFrame(columns=["service", "weekend_uplift_percent", "contract_section", "source_line"])
assert len(weekend_uplifts) == 0

print("Threshold premium rules:", len(threshold_premiums))
print("Daily quantity limit rules:", len(daily_caps))
print("Bundled delivery pairs:", len(bundles))
print("Volume discount rules:", len(volume_discounts), "across", volume_discounts["service"].nunique(), "services")
print("Exclusion window rules:", len(exclusion_windows))
print("Weekend/non-business-day uplift rules:", len(weekend_uplifts), "(contract Section 10 is '_None._')")

display(threshold_premiums)
display(daily_caps)
display(bundles)
display(volume_discounts)
display(exclusion_windows)


Threshold premium rules: 18
Daily quantity limit rules: 18
Bundled delivery pairs: 7
Volume discount rules: 4 across 3 services
Exclusion window rules: 15
Weekend/non-business-day uplift rules: 0 (contract Section 10 is '_None._')


,service,threshold_units,premium_percent,contract_section,source_line
0,Advanced Vascular Endoscopic Procedure,10,25,5,156
1,Ambulatory Gastrointestinal Nutritional Support,16,15,5,157
2,Assisted Cardiac Ventilation Support,12,30,5,158
3,Continuous Infectious Critical Care Occupancy,6,40,5,159
4,Elective Ophthalmic Radiotherapy Fraction,8,30,5,160
5,Emergency Rheumatologic Rehabilitation Programme,8,30,5,161
6,Extended Psychiatric Endoscopic Procedure,6,25,5,162
7,Intensive Paediatric Consultation,6,20,5,163
8,Intermittent Psychiatric Rehabilitation Programme,6,30,5,164
9,Outpatient Geriatric Diagnostic Imaging,12,15,5,165


,service,maximum_units_per_patient_day,contract_section,source_line
0,Advanced Orthopaedic Ward Bed Occupancy,12,6,185
1,Ambulatory Cardiac Ward Bed Occupancy,4,6,186
2,Ambulatory Immunologic Ward Bed Occupancy,4,6,187
3,Assisted Gastrointestinal Anaesthesia Administ...,24,6,188
4,Assisted Urologic Nursing Observation,4,6,189
5,Bedside Gastrointestinal Biopsy Procedure,8,6,190
6,Elective Gastrointestinal Nursing Observation,6,6,191
7,Extended Hepatic Transfusion Service,8,6,192
8,Extended Urologic Telemetry Monitoring,12,6,193
9,Focused Metabolic Biopsy Procedure,8,6,194


,service_a,bundled_rate_a_cents,service_b,bundled_rate_b_cents,contract_section,source_line
0,Ambulatory Obstetric Case Conference,11875,Focused Vascular Infusion Therapy,4900,7,212
1,Ambulatory Paediatric Critical Care Occupancy,3550,Ambulatory Rheumatologic Rehabilitation Programme,28750,7,213
2,Assisted Paediatric Physiotherapy Session,2675,Outpatient Orthopaedic Recovery Room Occupancy,9375,7,214
3,Continuous Psychiatric Rehabilitation Programme,28625,Postoperative Palliative Recovery Room Occupancy,70975,7,215
4,Elective Obstetric Transfusion Service,1250,Intermittent Rheumatologic Dialysis Session,111675,7,216
5,Emergency Cardiac Physiotherapy Session,8400,Extended Obstetric Case Conference,35200,7,217
6,Specialist Immunologic Consultation,91600,Supervised Rheumatologic Dialysis Session,13900,7,218


,service,threshold_units,discount_percent,contract_section,source_line
0,Ambulatory Musculoskeletal Ventilation Support,80,15,8,232
1,Ambulatory Musculoskeletal Ventilation Support,240,30,8,233
2,Standard Oncology Ward Bed Occupancy,120,10,8,234
3,Standard Orthopaedic Critical Care Occupancy,120,10,8,235


,service_a,window_days,service_b,contract_section,source_line
0,Advanced Paediatric Theatre Time,30,Bedside Cardiac Home Visit,9,247
1,Ambulatory Haematology Transfusion Service,7,Postoperative Geriatric Imaging Interpretation,9,248
2,Assisted Paediatric Theatre Time,10,Outpatient Cardiac Physiotherapy Session,9,249
3,Comprehensive Ophthalmic Laboratory Panel,7,Ambulatory Musculoskeletal Specimen Analysis,9,250
4,Continuous Gastrointestinal Transfusion Service,10,Postoperative Geriatric Case Conference,9,251
5,Focused Urologic Case Conference,21,Outpatient Urologic Endoscopic Procedure,9,252
6,Focused Vascular Transport Service,14,Bedside Cardiac Home Visit,9,253
7,Inpatient Psychiatric Home Visit,14,Extended Hepatic Discharge Planning,9,254
8,Intensive Rheumatologic Radiotherapy Fraction,10,Standard Paediatric Dialysis Session,9,255
9,Intermittent Neurological Laboratory Panel,30,Intermittent Urologic Telemetry Monitoring,9,256


### Service matching

Hospital 4's billing descriptions use the same abbreviation conventions, the same
adjective/specialty/service-class vocabulary, and the same free-text style as Hospital 1's
(`adv`, `asst`, `amb`, `interm`, `spcm`, `anly`, clinical-specialty abbreviations like
`card`, `rheum`, `vasc`, `ortho`, etc.) -- this is the same synthetic generator across
hospitals. The matcher, its alias tables, and its conservative acceptance thresholds
(`MIN_SCORE=0.55`, `GAP_THRESHOLD=0.08`) are therefore reused verbatim rather than
re-tuned, keeping the audit conservative in exactly the same way: unmatched or ambiguous
descriptions go to `needs_review` rather than being guessed or silently treated as
correct.


In [4]:
import difflib
import re
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd
from IPython.display import display

REF_CODE_RE = re.compile(r"/[A-Z]{1,4}-?\d+\s*$")

TOKEN_ALIASES = {
    # Generic billing abbreviations
    "adv": "advanced", "asst": "assisted", "amb": "ambulatory", "inpt": "inpatient",
    "outpt": "outpatient", "obs": "observation", "spclst": "specialist", "supv": "supervised",
    "rtn": "routine", "std": "standard", "cont": "continuous", "interm": "intermittent",
    "proc": "procedure", "procure": "procedure", "sess": "session", "svc": "service",
    "prog": "programme", "physio": "physiotherapy", "occ": "occupancy", "rm": "room",
    "cs": "case", "conf": "conference", "wd": "ward", "bd": "bed", "anaes": "anaesthesia",
    "anaest": "anaesthesia", "anest": "anaesthesia", "admin": "administration",
    "spcm": "specimen", "anly": "analysis", "pnl": "panel", "plng": "planning",
    "disp": "dispensing", "pharm": "pharmaceutical", "radiother": "radiotherapy",
    "radioth": "radiotherapy", "biop": "biopsy", "dial": "dialysis", "transf": "transfusion",
    "crit": "critical", "cr": "critical", "nutr": "nutritional", "wnd": "wound",
    "rehab": "rehabilitation", "vent": "ventilation", "tele": "telemetry", "img": "imaging",
    "diag": "diagnostic", "isom": "isolation", "paed": "paediatric", "paeds": "paediatric",
    # Clinical / specialty abbreviations
    "ent": "otolaryngologic", "gi": "gastrointestinal", "ortho": "orthopaedic",
    "ophth": "ophthalmic", "haem": "haematology", "hema": "haematology", "neuro": "neurological",
    "rheum": "rheumatologic", "urol": "urologic", "pulm": "pulmonary", "psych": "psychiatric",
    "ger": "geriatric", "onco": "oncology", "derm": "dermatologic", "hep": "hepatic",
    "vasc": "vascular", "endo": "endocrine", "obst": "obstetric", "immun": "immunologic",
    "infect": "infectious", "msk": "musculoskeletal", "musk": "musculoskeletal", "pall": "palliative",
    "card": "cardiac", "metab": "metabolic",
}

CLINICAL_ANCHORS = {
    "cardiac", "haematology", "infectious", "metabolic", "neurological", "rheumatologic",
    "ophthalmic", "immunologic", "musculoskeletal", "gastrointestinal", "geriatric", "urologic",
    "pulmonary", "psychiatric", "oncology", "otolaryngologic", "palliative", "renal", "vascular",
    "dermatologic", "paediatric", "endocrine", "obstetric", "hepatic", "orthopaedic"
}

SERVICE_CLASS_ALIASES = {
    "specimen": {"specimen", "analysis", "panel", "laboratory", "lab"},
    "analysis": {"specimen", "analysis", "panel", "laboratory", "lab"},
    "panel": {"specimen", "analysis", "panel", "laboratory", "lab"},
    "dialysis": {"dialysis"},
    "wound": {"wound", "care"},
    "rehabilitation": {"rehabilitation", "programme"},
    "consultation": {"consultation"},
    "transport": {"transport"},
    "occupancy": {"occupancy", "room", "bed", "ward"},
    "endoscopic": {"endoscopic", "procedure"},
    "radiotherapy": {"radiotherapy", "fraction"},
    "infusion": {"infusion", "therapy"},
    "discharge": {"discharge", "planning"},
    "pharmaceutical": {"pharmaceutical", "dispensing"},
    "ventilation": {"ventilation", "support"},
    "nutritional": {"nutritional", "support"},
    "transfusion": {"transfusion"},
    "sterilisation": {"sterilisation"},
    "theatre": {"theatre"},
    "critical": {"critical", "care"},
}

def normalize_description(text: str):
    text = REF_CODE_RE.sub("", str(text))
    text = text.replace("-", " ").replace("/", " ")
    tokens = re.findall(r"[A-Za-z]+", text.lower())
    return [TOKEN_ALIASES.get(t, t) for t in tokens]

def word_score(token: str, word: str) -> float:
    token, word = token.lower(), word.lower()
    if token == word:
        return 1.0
    if len(token) >= 3 and word.startswith(token):
        return 0.85 + 0.15 * (len(token) / len(word))
    it = iter(word)
    if len(token) >= 2 and all(ch in it for ch in token):
        return 0.45 + 0.35 * (len(token) / len(word))
    r = difflib.SequenceMatcher(None, token, word).ratio()
    return r * 0.55 if r > 0.75 else 0.0

def score_candidate(desc_tokens, service_words):
    pairs = sorted(
        ((word_score(t, w), i, j) for i, t in enumerate(desc_tokens) for j, w in enumerate(service_words)),
        reverse=True,
    )
    used_i, used_j, matched_score, matched_pairs = set(), set(), 0.0, 0
    for s, i, j in pairs:
        if s <= 0 or i in used_i or j in used_j:
            continue
        used_i.add(i); used_j.add(j)
        matched_score += s
        matched_pairs += 1
    n_words, n_tokens = len(service_words), len(desc_tokens)
    coverage_desc = matched_pairs / n_tokens if n_tokens else 0
    return (matched_score / n_words) * (0.6 + 0.4 * coverage_desc) if n_words else 0

def anchor_sets(tokens):
    anchors = {t for t in tokens if t in CLINICAL_ANCHORS}
    classes = set()
    token_set = set(tokens)
    for key, aliases in SERVICE_CLASS_ALIASES.items():
        if token_set & aliases:
            classes.add(key)
    return anchors, classes

def candidate_conflict(desc_tokens, service_tokens):
    d_anchor, d_class = anchor_sets(desc_tokens)
    s_anchor, s_class = anchor_sets(service_tokens)
    anchor_conflict = bool(d_anchor and s_anchor and d_anchor.isdisjoint(s_anchor))
    class_conflict = bool(d_class and s_class and d_class.isdisjoint(s_class))
    return anchor_conflict, class_conflict, d_anchor, d_class, s_anchor, s_class

def match_description(desc: str, schedule: pd.DataFrame, top_k=8):
    tokens = normalize_description(desc)
    results = []
    for _, row in schedule.iterrows():
        svc = row["service"]
        svc_tokens = normalize_description(svc)
        score = score_candidate(tokens, svc_tokens)
        anchor_conflict, class_conflict, d_anchor, d_class, s_anchor, s_class = candidate_conflict(tokens, svc_tokens)
        if anchor_conflict:
            score -= 0.40
        if class_conflict:
            score -= 0.20
        results.append({
            "score": score, "service": svc,
            "anchor_conflict": anchor_conflict, "class_conflict": class_conflict,
            "d_anchor": d_anchor, "d_class": d_class, "s_anchor": s_anchor, "s_class": s_class,
        })
    results.sort(key=lambda r: (r["score"], r["service"]), reverse=True)
    return results[:top_k]

GAP_THRESHOLD = 0.08
MIN_SCORE = 0.55

def half_up(value):
    return int(Decimal(value).to_integral_value(rounding=ROUND_HALF_UP))

def plausible_unit_prices(service):
    base = int(rate_schedule.loc[rate_schedule["service"] == service, "base_rate_cents"].iloc[0])
    prices = {base}
    for _, r in threshold_premiums.loc[threshold_premiums["service"] == service].iterrows():
        prices.add(half_up(Decimal(base) * (Decimal(100 + int(r["premium_percent"])) / 100)))
    for _, r in volume_discounts.loc[volume_discounts["service"] == service].iterrows():
        prices.add(half_up(Decimal(base) * (Decimal(100 - int(r["discount_percent"])) / 100)))
    return prices

# ---- Resolve every distinct billing description against the Hospital 4 rate schedule ----
observed_prices = line_items.groupby("description")["unit_price_cents"].apply(lambda s: set(int(x) for x in s.unique()))

match_records = []
for desc in line_items["description"].dropna().astype(str).unique():
    top = match_description(desc, rate_schedule, top_k=8)
    top1, top2 = top[0], (top[1] if len(top) > 1 else None)
    gap = top1["score"] - (top2["score"] if top2 else 0.0)
    obs = observed_prices.get(desc, set())
    structurally_compatible = [r for r in top if not r["anchor_conflict"] and not r["class_conflict"]]

    status, chosen, reason = "needs_review", None, ""
    if (not top1["anchor_conflict"] and not top1["class_conflict"]
            and top1["score"] >= MIN_SCORE and gap >= GAP_THRESHOLD):
        status, chosen, reason = "matched", top1["service"], "strong_text_match"
    else:
        close = [r for r in structurally_compatible if r["score"] >= max(top1["score"] - GAP_THRESHOLD, 0)]
        consistent = [r for r in close if obs and obs.issubset(plausible_unit_prices(r["service"]))]
        if len(consistent) == 1 and consistent[0]["score"] >= 0.35:
            status, chosen, reason = "resolved_by_price", consistent[0]["service"], "secondary_price_support"
        else:
            if top1["anchor_conflict"]:
                reason = "clinical_anchor_conflict"
            elif top1["class_conflict"]:
                reason = "service_class_conflict"
            elif len(structurally_compatible) > 1:
                reason = "ambiguous_text_match"
            else:
                reason = "insufficient_text_evidence"

    match_records.append({
        "description": desc, "matched_service": chosen, "text_score": round(top1["score"], 3),
        "gap": round(gap, 3), "match_status": status, "match_reason": reason,
        "top_candidates": str([(round(r["score"], 3), r["service"]) for r in top[:3]]),
    })

description_matches = pd.DataFrame(match_records)
print("Match status breakdown (distinct descriptions):")
print(description_matches["match_status"].value_counts())

descriptions_for_review = description_matches[description_matches["match_status"] == "needs_review"]
print(f"\n{len(descriptions_for_review)} descriptions sent for manual review:")
display(descriptions_for_review[["description", "text_score", "gap", "match_reason", "top_candidates"]])

line_items = line_items.merge(
    description_matches[["description", "matched_service", "match_status", "match_reason"]],
    on="description", how="left", validate="many_to_one",
)
line_items["service_date_raw"] = line_items["service_date"]
line_items["service_date"] = pd.to_datetime(line_items["service_date_raw"], format="%Y-%m-%d", errors="coerce")

print("\nLine items by match status:")
print(line_items["match_status"].value_counts())


Match status breakdown (distinct descriptions):
match_status
matched              508
needs_review          15
resolved_by_price     11
Name: count, dtype: int64

15 descriptions sent for manual review:


,description,text_score,gap,match_reason,top_candidates
17,CARDIAC physio SESS /CW-4455,0.750,0.000,ambiguous_text_match,"[(0.75, 'Outpatient Cardiac Physiotherapy Sess..."
294,RHEUM rehabilitation PROGRAMME /CW-4700,0.750,0.000,ambiguous_text_match,"[(0.75, 'Emergency Rheumatologic Rehabilitatio..."
386,SVC - ortho STERILISATION,0.183,0.150,insufficient_text_evidence,"[(0.183, 'Inpatient Orthopaedic Anaesthesia Ad..."
387,CONSULT rtn NEURO /CW-8916,0.183,0.000,ambiguous_text_match,"[(0.183, 'Intermittent Neurological Laboratory..."
491,ADMIN spclst NEURO anaes /CW-6871,0.175,0.000,ambiguous_text_match,"[(0.175, 'Intermittent Neurological Laboratory..."
495,RTN - ortho TRANSF service /CW-3818,0.175,0.175,insufficient_text_evidence,"[(0.175, 'Inpatient Orthopaedic Anaesthesia Ad..."
500,ASST - obstetric TRANSP svc /CW-7850,0.400,0.000,ambiguous_text_match,"[(0.4, 'Elective Obstetric Transfusion Service..."
501,VST foc NEURO /CW-8682,0.183,0.000,ambiguous_text_match,"[(0.183, 'Intermittent Neurological Laboratory..."
525,OUTPT metabolic PHARM disp,-0.025,0.000,service_class_conflict,"[(-0.025, 'Focused Metabolic Biopsy Procedure'..."
527,SPCLST wnd CR /CW-6542,0.183,0.000,ambiguous_text_match,"[(0.183, 'Specialist Gastrointestinal Imaging ..."



Line items by match status:
match_status
matched              10299
resolved_by_price      200
needs_review            61
Name: count, dtype: int64


### Occurrence mapping for reused invoice IDs

Hospital 4 reuses 5 invoice IDs across two genuinely different transactions each (same
pattern as Hospital 1): the line-ID group prefix (`H4-Lnnnnn-nn`) identifies the original
transaction group, which is mapped chronologically onto the invoice-table occurrences and
cross-checked against each occurrence's billed total.


In [5]:
from IPython.display import display

# Preserve original line-group identity. Synthetic Hospital 4 line IDs retain the
# transaction group even when an invoice identifier is accidentally reused.
line_items["line_group_num"] = (
    line_items["line_id"].astype(str)
    .str.extract(r"^H4-L(\d+)-", expand=False)
    .astype("Int64")
)
if line_items["line_group_num"].isna().any():
    raise ValueError("Unable to recover line-group identity from one or more line IDs")

# Invoice occurrence rank: chronological within reused invoice IDs.
invoices["_row_order"] = range(len(invoices))
invoices["occurrence_rank"] = (
    invoices.sort_values(["invoice_id", "invoice_date", "_row_order"])
    .groupby("invoice_id").cumcount()
)

# Line-group rank: chronological source-group order.
group_meta = (
    line_items.groupby(["invoice_id", "line_group_num"], dropna=False)
    .agg(group_billed_total_cents=("line_total_cents", "sum"),
         group_min_service_raw=("service_date", "first"))
    .reset_index()
)
group_meta["group_rank"] = (
    group_meta.sort_values(["invoice_id", "line_group_num"])
    .groupby("invoice_id").cumcount()
)

line_items = line_items.merge(
    group_meta[["invoice_id", "line_group_num", "group_rank", "group_billed_total_cents"]],
    on=["invoice_id", "line_group_num"], how="left",
)
line_items["occurrence_rank"] = line_items["group_rank"].astype(int)

occ_cols = ["invoice_id", "occurrence_rank", "invoice_date", "patient_id",
            "facility_code", "plan_tier", "contract_number", "invoice_total_cents"]
line_items = line_items.merge(
    invoices[occ_cols], on=["invoice_id", "occurrence_rank"], how="left",
    validate="many_to_one", suffixes=("", "_invoice"),
)

line_items["invoice_occurrence_key"] = (
    line_items["invoice_id"].astype(str) + "::" + line_items["occurrence_rank"].astype(str)
)

# Validate reused-ID mapping against billed totals when possible.
occ_validation = (
    line_items.groupby(["invoice_id", "occurrence_rank"], as_index=False)
    .agg(group_lines_billed_cents=("line_total_cents", "sum"))
    .merge(invoices[occ_cols], on=["invoice_id", "occurrence_rank"], how="left")
)
occ_validation["billed_group_matches_invoice"] = (
    occ_validation["group_lines_billed_cents"] == occ_validation["invoice_total_cents"]
)

print("Reused invoice IDs:", int(invoices["invoice_id"].duplicated(keep=False).sum()), "invoice rows,",
      invoices.loc[invoices["invoice_id"].duplicated(keep=False), "invoice_id"].nunique(), "distinct IDs")
print("Occurrence groups whose billed line sum exactly matches the invoice record:",
      int(occ_validation["billed_group_matches_invoice"].sum()), "/", len(occ_validation))
display(occ_validation.loc[~occ_validation["billed_group_matches_invoice"],
                            ["invoice_id", "occurrence_rank", "group_lines_billed_cents", "invoice_total_cents"]])


Reused invoice IDs: 10 invoice rows, 5 distinct IDs
Occurrence groups whose billed line sum exactly matches the invoice record: 834 / 840


,invoice_id,occurrence_rank,group_lines_billed_cents,invoice_total_cents
5,INV-H4-000006,0,641175,640175
158,INV-H4-000157,0,1041600,1066600
324,INV-H4-000323,0,1905175,1906175
484,INV-H4-000483,0,1602575,1702575
505,INV-H4-000504,0,970875,995875
554,INV-H4-000554,0,3583010,3608010


### Section 11 checks + unit basis, daily cap, duplicate and exclusion checks

* **11.1** contract-number mismatch, and invoice IDs unique across the term (reused IDs);
* **11.2** invalid / out-of-term / service-date-after-invoice-date;
* **11.3** the same Service billed twice for the same Patient + Service Date, on one
  invoice or across several -- clause 11.3 explicitly ties this to the Section 6.2
  aggregation rule ("irrespective of the number of line items or invoices"). The later
  occurrence (by invoice date, then line ID) is treated as the non-billable duplicate,
  the same convention Hospital 1 used.

Two further contract constraints are checked before pricing: Section 6 daily quantity
limits (18 named services only) and Section 3 billed unit basis vs. the contract unit.
Section 9 exclusion windows remove the excluded (non-billable) service line from the
expected reimbursement, exactly as Hospital 1's Section 10 did.


In [6]:
import pandas as pd

# --- 11.1 invoice-level checks ---
invoices["contract_number_mismatch"] = invoices["contract_number"] != CONTRACT_NUMBER
invoices["duplicate_invoice_id"] = invoices["invoice_id"].duplicated(keep=False)

# --- 11.2 service dates ---
line_items["malformed_service_date"] = line_items["service_date"].isna()
line_items["service_date_out_of_window"] = (
    (~line_items["malformed_service_date"])
    & ((line_items["service_date"] < CONTRACT_START) | (line_items["service_date"] > CONTRACT_END))
)
line_items["service_date_after_invoice_date"] = (
    (~line_items["malformed_service_date"])
    & (line_items["service_date"] > line_items["invoice_date"])
)

# --- billed unit basis vs contract unit (Section 3) ---
CONTRACT_UNIT_TO_BILLED = {
    "per hour": "per_hour", "per day of service": "per_day", "per visit": "per_visit",
    "per test": "per_test", "per procedure": "per_procedure", "per night of occupancy": "per_night",
    "per item supplied": "per_item", "per unit dispensed": "per_unit_dispensed",
    "per hour, per item": "per_hour_per_item",
}
_unit_map = rate_schedule.set_index("service")["contract_unit"].map(CONTRACT_UNIT_TO_BILLED).to_dict()
line_items["expected_unit_basis"] = line_items["matched_service"].map(_unit_map)
line_items["wrong_unit_basis"] = (
    line_items["expected_unit_basis"].notna()
    & (line_items["expected_unit_basis"] != line_items["unit_basis_as_billed"])
)

# --- Section 6 daily quantity limits: cap applied across all lines for patient+service+Service Day ---
_cap_map = daily_caps.set_index("service")["maximum_units_per_patient_day"].to_dict()
_valid_priced = line_items.dropna(subset=["matched_service", "service_date", "patient_id"]).copy()
_valid_priced = _valid_priced.sort_values(["patient_id", "matched_service", "service_date", "line_id"])
_valid_priced["_day_qty_prior"] = _valid_priced.groupby(
    ["patient_id", "matched_service", "service_date"]
)["quantity"].cumsum() - _valid_priced["quantity"]
_valid_priced["_cap"] = _valid_priced["matched_service"].map(_cap_map)
_valid_priced["billable_quantity"] = _valid_priced["quantity"]
_mask_capped = _valid_priced["_cap"].notna()
_valid_priced.loc[_mask_capped, "billable_quantity"] = (
    _valid_priced.loc[_mask_capped, "quantity"]
    .where(_valid_priced.loc[_mask_capped, "_day_qty_prior"] < _valid_priced.loc[_mask_capped, "_cap"], 0)
    .clip(upper=_valid_priced.loc[_mask_capped, "_cap"] - _valid_priced.loc[_mask_capped, "_day_qty_prior"])
    .clip(lower=0)
)
line_items["_day_qty_prior"] = pd.NA
line_items["billable_quantity"] = line_items["quantity"]
line_items.loc[_valid_priced.index, "_day_qty_prior"] = _valid_priced["_day_qty_prior"]
line_items.loc[_valid_priced.index, "billable_quantity"] = _valid_priced["billable_quantity"]
line_items["daily_cap_exceeded"] = line_items["billable_quantity"] < line_items["quantity"]

# --- 11.3 duplicate patient/service/date billing (one invoice or across several) ---
line_items["cross_invoice_duplicate"] = False
_dup_valid = _valid_priced[_valid_priced["matched_service"].notna()].copy()
for _, grp in _dup_valid.groupby(["patient_id", "matched_service", "service_date"], dropna=False):
    if len(grp) < 2:
        continue
    ordered = grp.sort_values(["invoice_date", "line_id"])
    line_items.loc[ordered.index[1:], "cross_invoice_duplicate"] = True

# --- Section 9 exclusion windows ---
line_items["exclusion_violation"] = False
for _, rule in exclusion_windows.iterrows():
    a_rows = _valid_priced[_valid_priced["matched_service"] == rule["service_a"]]
    b_rows = _valid_priced[_valid_priced["matched_service"] == rule["service_b"]]
    if a_rows.empty or b_rows.empty:
        continue
    merged = a_rows[["patient_id", "service_date"]].reset_index().merge(
        b_rows[["patient_id", "service_date"]].reset_index(),
        on="patient_id", suffixes=("_a", "_b"),
    )
    diff_days = (merged["service_date_a"] - merged["service_date_b"]).abs().dt.days
    violations = merged[diff_days <= int(rule["window_days"])].copy()
    if not violations.empty:
        line_items.loc[violations["index_a"], "exclusion_violation"] = True

# --- billed line arithmetic, an explicit audit check independent of contract pricing ---
line_items["line_total_arithmetic"] = (
    line_items["line_total_cents"] != line_items["unit_price_cents"] * line_items["quantity"]
)

LINE_CHECK_COLS = [
    "malformed_service_date", "service_date_out_of_window", "service_date_after_invoice_date",
    "wrong_unit_basis", "daily_cap_exceeded", "cross_invoice_duplicate", "exclusion_violation",
    "line_total_arithmetic"
]

for col in LINE_CHECK_COLS:
    print(f"{col}: {int(line_items[col].sum())} line items")
print("\ncontract_number_mismatch:", int(invoices["contract_number_mismatch"].sum()), "invoice rows")
print("duplicate_invoice_id:", int(invoices["duplicate_invoice_id"].sum()), "invoice rows")


malformed_service_date: 6 line items
service_date_out_of_window: 6 line items
service_date_after_invoice_date: 8 line items
wrong_unit_basis: 12 line items
daily_cap_exceeded: 6 line items
cross_invoice_duplicate: 5 line items
exclusion_violation: 5 line items
line_total_arithmetic: 6 line items

contract_number_mismatch: 6 invoice rows
duplicate_invoice_id: 10 invoice rows


### Pricing engine

Clause 4.1's stated order -- bundle -> facility -> plan -> premium/uplift -> cumulative
discount -- is identical to Hospital 1's Section 3.2(a)-(e) order, so the same staged
pricing engine is reused. Facility and plan multipliers are 1x throughout (clause 2.2).
Rounding is half-up-cent, applied after each stage (clause 4.2), matching Hospital 1's
`half_up` helper used at every step. Threshold premiums (Section 5) apply to the full
per-Service-Day quantity once the aggregate exceeds the stated threshold, mirroring
Hospital 1's Section 5 mechanism. Discounts (Section 8) are cumulative and hospital-wide
across the whole contract term, counted in Service-Date-then-line-ID order (clause 8.5),
with the discount applying strictly *after* the line on which the threshold is crossed
(clause 8.4) -- the same "prior-to-line threshold" rule Hospital 1 used for its Section 7
discounts.

An unmatched service or a malformed service date carries the billed line total through as
a **provisional** expected amount (never a dropped or invented figure), with confidence
lowered downstream -- the same policy as Hospital 1.


In [7]:
from decimal import Decimal
import pandas as pd
from IPython.display import display

# Clause 4.1 order: (a) bundle -> (b) facility -> (c) plan -> (d) premium/uplift -> (e) discount
base_rate_map = rate_schedule.set_index("service")["base_rate_cents"].to_dict()
line_items["base_rate_cents"] = line_items["matched_service"].map(base_rate_map)

# --- (a) Section 7 bundle substitution ---
bundle_map_a = {row.service_a: (row.service_b, row.bundled_rate_a_cents) for row in bundles.itertuples()}
bundle_map_b = {row.service_b: (row.service_a, row.bundled_rate_b_cents) for row in bundles.itertuples()}

_services_present = (
    line_items.dropna(subset=["matched_service", "patient_id", "service_date"])
    .groupby(["patient_id", "service_date"])["matched_service"]
    .apply(set)
)

def _bundle_rate(row):
    svc, key = row["matched_service"], (row["patient_id"], row["service_date"])
    present = _services_present.get(key, set())
    if svc in bundle_map_a and bundle_map_a[svc][0] in present:
        return bundle_map_a[svc][1]
    if svc in bundle_map_b and bundle_map_b[svc][0] in present:
        return bundle_map_b[svc][1]
    return row["base_rate_cents"]

line_items["rate_after_bundle"] = line_items.apply(_bundle_rate, axis=1)

# --- (b)/(c) facility & plan-tier multipliers: 1x for Hospital 4 (clause 2.2) ---
line_items["rate_after_facility_plan"] = line_items["rate_after_bundle"]

# --- (d.1) Section 5 threshold premium, aggregate quantity per Service Day ---
_premium_map = threshold_premiums.set_index("service")[["threshold_units", "premium_percent"]].to_dict("index")
line_items["_day_qty_this_service"] = line_items.groupby(
    ["patient_id", "matched_service", "service_date"]
)["quantity"].transform("sum")

line_items["premium_applies"] = False
line_items["rate_after_premium"] = line_items["rate_after_facility_plan"]
for idx, row in line_items.iterrows():
    rule = _premium_map.get(row["matched_service"])
    rate = row["rate_after_facility_plan"]
    if pd.isna(rate) or rule is None or pd.isna(row["_day_qty_this_service"]):
        continue
    if row["_day_qty_this_service"] > rule["threshold_units"]:
        line_items.at[idx, "premium_applies"] = True
        line_items.at[idx, "rate_after_premium"] = half_up(
            Decimal(rate) * (Decimal(100 + int(rule["premium_percent"])) / 100)
        )

# --- (d.2) Section 10 non-business-day uplift: contract states "_None._" -> never applies ---
_weekend_map = weekend_uplifts.set_index("service")["weekend_uplift_percent"].to_dict() if len(weekend_uplifts) else {}
_is_weekend = line_items["service_date"].dt.dayofweek >= 5
line_items["weekend_applies"] = False
line_items["rate_after_weekend"] = line_items["rate_after_premium"]
for idx, (r, s, w) in enumerate(zip(line_items["rate_after_premium"], line_items["matched_service"], _is_weekend)):
    pct = _weekend_map.get(s)
    if pd.isna(r) or pct is None or pd.isna(w) or not w:
        continue
    line_items.at[line_items.index[idx], "weekend_applies"] = True
    line_items.at[line_items.index[idx], "rate_after_weekend"] = half_up(Decimal(r) * (Decimal(100 + int(pct)) / 100))

# --- (e) Section 8 cumulative discount: hospital-wide, Service Date then line_id (clause 8.5) ---
line_items["_discount_pct"] = 0
for service, _ in volume_discounts.groupby("service"):
    idx = line_items.index[line_items["matched_service"] == service]
    if len(idx) == 0:
        continue
    sub = line_items.loc[idx].sort_values(["service_date", "line_id"])
    cumulative_before = sub["quantity"].cumsum() - sub["quantity"]
    tiers = volume_discounts[volume_discounts["service"] == service].sort_values("threshold_units")
    pct = pd.Series(0, index=sub.index, dtype="int64")
    for _, tier in tiers.iterrows():
        pct = pct.where(cumulative_before <= int(tier["threshold_units"]), int(tier["discount_percent"]))
    line_items.loc[sub.index, "_discount_pct"] = pct

def _apply_discount(rate, pct):
    if pd.isna(rate) or not pct:
        return rate
    return half_up(Decimal(rate) * (Decimal(100 - int(pct)) / 100))

line_items["effective_unit_rate_cents"] = [
    _apply_discount(r, p) for r, p in zip(line_items["rate_after_weekend"], line_items["_discount_pct"])
]

# Keep billed line totals as a provisional amount when the contract basis is not known.
line_items["expected_line_total_provisional"] = False
line_items["expected_line_total_cents"] = (
    line_items["effective_unit_rate_cents"] * line_items["billable_quantity"].astype("float64")
)
_provisional = line_items["matched_service"].isna() | line_items["malformed_service_date"]
line_items.loc[_provisional, "expected_line_total_cents"] = line_items.loc[_provisional, "line_total_cents"]
line_items.loc[_provisional, "expected_line_total_provisional"] = True

# Section 9 excluded service_a lines are not billable.
line_items.loc[line_items["exclusion_violation"], "expected_line_total_cents"] = 0
line_items.loc[line_items["exclusion_violation"], "expected_line_total_provisional"] = False

# Clause 11.3 later duplicate of same patient + service + Service Day is not separately reimbursable.
line_items.loc[line_items["cross_invoice_duplicate"], "expected_line_total_cents"] = 0
line_items.loc[line_items["cross_invoice_duplicate"], "expected_line_total_provisional"] = False

# --- Diagnostic categories: compare billed rate against each pricing stage ---
line_items["bundle_applies"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["matched_service"] in bundle_map_a or r["matched_service"] in bundle_map_b)
        and ((r["patient_id"], r["service_date"]) in _services_present.index)
        and (
            (r["matched_service"] in bundle_map_a and bundle_map_a[r["matched_service"]][0] in _services_present.get((r["patient_id"], r["service_date"]), set()))
            or (r["matched_service"] in bundle_map_b and bundle_map_b[r["matched_service"]][0] in _services_present.get((r["patient_id"], r["service_date"]), set()))
        )
    ), axis=1
)
line_items["bundle_not_applied"] = (
    line_items["bundle_applies"]
    & (line_items["unit_price_cents"] == line_items["base_rate_cents"])
    & (line_items["rate_after_bundle"] != line_items["base_rate_cents"])
)

line_items["unit_price_mismatch"] = (
    line_items["matched_service"].notna()
    & line_items["effective_unit_rate_cents"].notna()
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)

_premium_rate_options = {}
for service, rows in threshold_premiums.groupby("service"):
    base = int(base_rate_map[service])
    _premium_rate_options[service] = {half_up(Decimal(base) * (Decimal(100 + int(p)) / 100)) for p in rows["premium_percent"]}
line_items["premium_omitted"] = (
    line_items["premium_applies"]
    & (line_items["unit_price_cents"] == line_items["rate_after_facility_plan"])
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)
line_items["premium_incorrectly_applied"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["unit_price_cents"] != r["effective_unit_rate_cents"])
        and (
            (bool(r["premium_applies"]) and not bool(r["premium_omitted"])
             and r["unit_price_cents"] in _premium_rate_options.get(r["matched_service"], set()))
            or (not bool(r["premium_applies"])
                and int(r["unit_price_cents"]) in _premium_rate_options.get(r["matched_service"], set()))
        )
    ), axis=1
)

# Weekend uplift diagnostics: never populated for Hospital 4 (Section 10 is empty),
# columns kept for schema parity with the Hospital 1 pipeline.
line_items["weekend_uplift_omitted"] = False
line_items["weekend_uplift_incorrectly_applied"] = False

_discount_rate_options = {}
for service, rows in volume_discounts.groupby("service"):
    base = int(base_rate_map[service])
    _discount_rate_options[service] = {half_up(Decimal(base) * (Decimal(100 - int(p)) / 100)) for p in rows["discount_percent"]}
line_items["volume_discount_omitted"] = (
    (line_items["_discount_pct"] > 0)
    & (line_items["unit_price_cents"] == line_items["rate_after_weekend"])
    & (line_items["unit_price_cents"] != line_items["effective_unit_rate_cents"])
)
line_items["volume_discount_incorrectly_applied"] = line_items.apply(
    lambda r: (
        pd.notna(r["matched_service"])
        and (r["unit_price_cents"] != r["effective_unit_rate_cents"])
        and (
            ((int(r["_discount_pct"]) > 0) and not bool(r["volume_discount_omitted"]))
            or ((int(r["_discount_pct"]) == 0) and int(r["unit_price_cents"]) in _discount_rate_options.get(r["matched_service"], set()))
        )
    ), axis=1
)

print("Bundle applied:", int(line_items["bundle_applies"].sum()), "lines")
print("Premium applied:", int(line_items["premium_applies"].sum()), "lines")
print("Weekend uplift applied:", int(line_items["weekend_applies"].sum()), "lines (contract has none)")
print("Discount applied:", int((line_items["_discount_pct"] > 0).sum()), "lines")
print("Provisional expected lines:", int(line_items["expected_line_total_provisional"].sum()), "lines")

display(line_items[[
    "line_id", "invoice_id", "invoice_occurrence_key", "matched_service", "quantity",
    "billable_quantity", "base_rate_cents", "effective_unit_rate_cents",
    "expected_line_total_cents", "line_total_cents"
]].head())


Bundle applied: 1318 lines
Premium applied: 254 lines
Weekend uplift applied: 0 lines (contract has none)
Discount applied: 250 lines
Provisional expected lines: 67 lines


,line_id,invoice_id,invoice_occurrence_key,matched_service,quantity,billable_quantity,base_rate_cents,effective_unit_rate_cents,expected_line_total_cents,line_total_cents
0,H4-L00001-01,INV-H4-000001,INV-H4-000001::0,Intensive Rheumatologic Radiotherapy Fraction,11,11,23425.0,23425.0,257675.0,257675
1,H4-L00001-02,INV-H4-000001,INV-H4-000001::0,Intensive Urologic Specimen Analysis,12,12,19675.0,19675.0,236100.0,236100
2,H4-L00001-03,INV-H4-000001,INV-H4-000001::0,Advanced Orthopaedic Ward Bed Occupancy,3,3,97925.0,97925.0,293775.0,293775
3,H4-L00001-04,INV-H4-000001,INV-H4-000001::0,Intensive Vascular Sterilisation Service,5,5,25350.0,25350.0,126750.0,126750
4,H4-L00001-05,INV-H4-000001,INV-H4-000001::0,Preoperative Haematology Imaging Interpretation,7,7,11375.0,11375.0,79625.0,79625


### Aggregate line totals to invoice totals, and final submission output

Aggregation is at invoice-occurrence level first (so a reused invoice ID's two
transactions are priced independently), then collapsed to one row per invoice ID by
selecting the later occurrence -- identical to Hospital 1's convention. Confidence is a
deterministic audit-policy score, not a probability, using the same rule set as Hospital
1: unknown-service caps confidence at 0.35, a provisional expected total or a duplicate
invoice ID caps it at 0.55, a "corrective-rate" category (unit price / premium / discount
mismatch) caps it at 0.80, and two or more simultaneous error categories cap it at 0.75.


In [ ]:
from IPython.display import display

# Aggregate at invoice-occurrence level first, then collapse reused IDs to one prediction row.
occ_line = (
    line_items.groupby("invoice_occurrence_key").agg(
        expected_total_cents=("expected_line_total_cents", "sum"),
        any_provisional_expected=("expected_line_total_provisional", "any"),
        any_unmatched=("matched_service", lambda s: s.isna().any()),
        billed_line_total_sum_cents=("line_total_cents", "sum"),
    ).reset_index()
)
occ_line[["invoice_id", "occurrence_rank"]] = occ_line["invoice_occurrence_key"].str.split("::", expand=True)
occ_line["occurrence_rank"] = occ_line["occurrence_rank"].astype(int)

occ_flags = (
    line_items.groupby("invoice_occurrence_key")[LINE_CHECK_COLS + [
        "bundle_not_applied", "premium_omitted", "premium_incorrectly_applied",
        "volume_discount_omitted", "volume_discount_incorrectly_applied",
        "unit_price_mismatch", "weekend_uplift_omitted", "weekend_uplift_incorrectly_applied"
    ]].any().reset_index()
)

occ_invoice = invoices[
    ["invoice_id", "occurrence_rank", "invoice_date", "patient_id", "contract_number",
     "invoice_total_cents", "contract_number_mismatch", "duplicate_invoice_id"]
].copy()
occ_invoice["invoice_occurrence_key"] = (
    occ_invoice["invoice_id"].astype(str) + "::" + occ_invoice["occurrence_rank"].astype(str)
)

occ_summary = (
    occ_invoice.merge(occ_line, on=["invoice_occurrence_key", "invoice_id", "occurrence_rank"], how="left")
    .merge(occ_flags, on="invoice_occurrence_key", how="left")
)

# Explicit invoice-total arithmetic mismatch (clause 4.4: invoice total is the sum of line
# totals and nothing else).
occ_summary["invoice_total_mismatch"] = (
    occ_summary["billed_line_total_sum_cents"] != occ_summary["invoice_total_cents"]
)

# Reused invoice IDs: report the later invoice occurrence (same convention as Hospital 1).
latest_rank = occ_summary.groupby("invoice_id")["occurrence_rank"].transform("max")
occ_summary["selected_occurrence"] = occ_summary["occurrence_rank"] == latest_rank

PRED_CAT_COLS = [
    "malformed_service_date", "service_date_out_of_window", "service_date_after_invoice_date",
    "wrong_unit_basis", "daily_cap_exceeded", "cross_invoice_duplicate", "exclusion_violation",
    "line_total_arithmetic", "bundle_not_applied", "unit_price_mismatch", "premium_omitted",
    "premium_incorrectly_applied", "weekend_uplift_omitted", "weekend_uplift_incorrectly_applied",
    "volume_discount_omitted", "volume_discount_incorrectly_applied", "invoice_total_mismatch",
    "contract_number_mismatch",
    "duplicate_invoice_id"
]

def occurrence_error_category(row):
    cats = [c for c in PRED_CAT_COLS if bool(row.get(c, False))]
    if bool(row.get("any_unmatched", False)):
        cats.append("unknown_service")
    return "|".join(dict.fromkeys(cats))

def occurrence_confidence(row):
    cats = [c for c in PRED_CAT_COLS if bool(row.get(c, False))]
    has_unknown = bool(row.get("any_unmatched", False))
    confidence = 0.95 if not cats and not has_unknown else 0.90
    if has_unknown:
        confidence = min(confidence, 0.35)
    if bool(row.get("duplicate_invoice_id", False)):
        confidence = min(confidence, 0.55)
    if bool(row.get("any_provisional_expected", False)):
        confidence = min(confidence, 0.55)
    if any(c in {"unit_price_mismatch", "premium_incorrectly_applied",
               "volume_discount_incorrectly_applied", "weekend_uplift_incorrectly_applied"}
           for c in cats):
        confidence = min(confidence, 0.80)
    if len(cats) >= 2:
        confidence = min(confidence, 0.75)
    return round(confidence, 2)

occ_summary["error_category"] = occ_summary.apply(occurrence_error_category, axis=1)
occ_summary["confidence"] = occ_summary.apply(occurrence_confidence, axis=1)
occ_summary["flagged"] = (occ_summary["error_category"] != "").astype(int)

invoice_summary = occ_summary[occ_summary["selected_occurrence"]].copy()
invoice_summary = invoice_summary.sort_values(["invoice_id"]).reset_index(drop=True)

invoice_summary["billed_total_cents"] = invoice_summary["invoice_total_cents"]
invoice_summary["expected_total_cents"] = invoice_summary["expected_total_cents"].round().astype("Int64")

print("Flagged counts:", invoice_summary["flagged"].value_counts().to_dict())
print("\nConfidence distribution:")
print(invoice_summary["confidence"].value_counts().sort_index())
print("\nInvoices with provisional expected totals:", int(invoice_summary["any_provisional_expected"].sum()))
print("\nTop error categories among flagged invoices:")
display(invoice_summary.loc[invoice_summary["flagged"] == 1, "error_category"].value_counts().head(15))

# Hospital 4 output in the exact repository submission schema.
submission_hospital_4 = invoice_summary[
    ["invoice_id", "flagged", "error_category", "expected_total_cents", "billed_total_cents", "confidence"]
].copy()

required_columns = [
    "invoice_id", "flagged", "error_category",
    "expected_total_cents", "billed_total_cents", "confidence"
]
assert submission_hospital_4.columns.tolist() == required_columns
assert submission_hospital_4["invoice_id"].is_unique
assert submission_hospital_4["flagged"].isin([0, 1]).all()
assert submission_hospital_4["confidence"].between(0, 1).all()
assert len(submission_hospital_4) == invoices["invoice_id"].nunique()

output_path = REPO_ROOT / "submission.csv"
submission_hospital_4.to_csv(output_path, index=False)
print(f"\nWrote {len(submission_hospital_4)} rows to {output_path}")
display(submission_hospital_4.head(10))


Flagged counts: {0: 729, 1: 106}

Confidence distribution:
confidence
0.35     56
0.55      7
0.75     30
0.80      1
0.90     12
0.95    729
Name: count, dtype: int64

Invoices with provisional expected totals: 58

Top error categories among flagged invoices:


error_category
unit_price_mismatch|unknown_service                        33
unknown_service                                            13
unit_price_mismatch|premium_omitted                         4
line_total_arithmetic                                       3
duplicate_invoice_id                                        3
wrong_unit_basis                                            2
invoice_total_mismatch                                      2
unit_price_mismatch|volume_discount_incorrectly_applied     2
service_date_after_invoice_date                             2
malformed_service_date|unknown_service                      2
daily_cap_exceeded                                          2
unit_price_mismatch|premium_incorrectly_applied             2
cross_invoice_duplicate|unknown_service                     2
unit_price_mismatch|volume_discount_omitted                 2
service_date_after_invoice_date|unit_price_mismatch         1
Name: count, dtype: int64


Wrote 835 rows to C:\Users\AMER\Downloads\17Sep\insurance_auditing-main\hospital_4_predictions.csv


,invoice_id,flagged,error_category,expected_total_cents,billed_total_cents,confidence
0,INV-H4-000001,0,,1684325,1684325,0.95
1,INV-H4-000002,1,unit_price_mismatch|unknown_service,3722200,3709750,0.35
2,INV-H4-000003,1,wrong_unit_basis,2990225,2990225,0.90
3,INV-H4-000004,0,,4541425,4541425,0.95
4,INV-H4-000005,0,,1061450,1061450,0.95
5,INV-H4-000006,1,invoice_total_mismatch,641175,640175,0.90
6,INV-H4-000007,0,,2396750,2396750,0.95
7,INV-H4-000008,1,service_date_after_invoice_date|unit_price_mis...,3259925,3525273,0.75
8,INV-H4-000009,0,,3544989,3544989,0.95
9,INV-H4-000010,0,,4801591,4801591,0.95


## Hospital 4 decision log

**Contract-structure mapping.** Hospital 4's `conditional_reimbursement_agreement.md`
numbers its sections differently from Hospital 1's `provider_services_agreement.md`
(Base Rates is Section 3 not 4, Threshold Premiums is still 5, Daily Quantity Limits is 6
not 8, Bundled Delivery is 7 not 9, Discounts is 8 not 7, Exclusion Windows is 9 not 10,
Non-Business-Day Uplifts is 10 not 6). Each section was re-extracted from the actual
clause headers rather than assumed to line up positionally with Hospital 1's, and the
extracted row counts were asserted against a manual read of the contract (98 base rates,
18 threshold premiums, 18 daily caps, 7 bundle pairs, 4 discount tiers across 3 services,
15 exclusion windows, 0 weekend uplifts).

**No weekend/non-business-day uplift.** Section 10 of Hospital 4's agreement is the single
word "_None._". The weekend-uplift columns and pricing step are kept in the code for
schema parity with Hospital 1's pipeline but structurally never fire, since the uplift
table is empty.

**Plan tier is not an audit check for Hospital 4.** Hospital 1's contract named a closed
set of valid tiers (clause 1.3); Hospital 4's contract (clause 2.2) only says plan tier
does not affect the rate. Absent a stated closed set, an unusual plan-tier value is not
treated as a contract violation here -- this is a reading I could not verify further from
the contract text alone.

**Clause 11.3 duplicate-billing interpretation.** 11.3 prohibits billing the same Service
twice for the same Patient + Service Date "whether on one invoice or across several," and
ties this explicitly to the Section 6.2 aggregation principle. I read this as a general
no-double-billing rule for *every* service (not only the 18 services with a stated
Section 6 numeric cap), and applied the same later-occurrence-is-the-duplicate convention
Hospital 1 used. This is the most consequential ambiguity call in this pass: an
alternative reading would confine 11.3 to only the capped services, which would leave the
5 flagged cross-invoice duplicates on uncapped services unflagged.

**Known limitation -- bundle rates are not part of the service-matcher's secondary price
evidence.** `plausible_unit_prices()` (used to break text-matching ties) only considers
base/premium/discount-derived prices, not bundle-substituted rates. Inspecting
`INV-H4-000002`, the ambiguous description "CARDIAC physio SESS" ties between "Outpatient
Cardiac Physiotherapy Session" and "Emergency Cardiac Physiotherapy Session"; its billed
price (GBP 84.00) matches the Section 7 *bundled* rate for "Emergency Cardiac
Physiotherapy Session" exactly (and that invoice also bills "Extended Obstetric Case
Conference," its bundle partner, on the same date) -- strong secondary evidence the
matcher does not use. The line is conservatively left `needs_review` rather than resolved,
consistent with the "flag, don't guess" policy, but it means bundle-linked ambiguous
descriptions are systematically under-resolved. This carries over unchanged from Hospital
1, which has the same gap.

**Reused invoice IDs and occurrence mapping.** Identical convention to Hospital 1: the
line-ID group prefix recovers the original transaction group, mapped chronologically onto
invoice-table occurrences, and the later occurrence is the one reported in the
one-row-per-invoice submission.

**No labelled evaluation for Hospital 4.** Only Hospital 1 has ground-truth labels in this
exercise; Hospital 4's predictions could not be scored against labels and are not
recalibrated to any Hospital-4-specific ground truth. The method and its calibration are
carried over from the Hospital 1 development pass.
